# 01. Optuna sweep for ModernBERT-base on the 108th Congress

Searches the settings that model T uses for every Congress (`docs/preregistration.md` §5): 15 TPE trials, each trained on the 108th Congress's train split and scored on its validation split. The test split is not used here. Each trial is a nested MLflow run under one parent run in the experiment `congress-speech`. The Optuna study is kept on Google Drive, so a disconnected session resumes where it stopped.

**Before running:**

1. Runtime → Change runtime type → **L4 GPU**.
2. Colab Secrets: `MLFLOW_USER`, `MLFLOW_PASSWORD`, `HF_TOKEN`, and `GITHUB_TOKEN` (while the repository is private).

Expect about 2.5 hours (10 minutes per trial), about 12 compute units at the L4 rate.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L

## 2. Get the code
The repository is private until the post is published, so the clone uses the `GITHUB_TOKEN` secret if it is set.

In [ ]:
import os, subprocess, sys
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret("GITHUB_TOKEN")
url = f"https://{'kvenanzi:' + token + '@' if token else ''}github.com/kvenanzi/congress-speech.git"   # fine-grained PAT, Contents: read
if not os.path.isdir("congress-speech"):
    subprocess.run(["git", "clone", "-q", url], check=True)
else:
    subprocess.run(["git", "-C", "congress-speech", "pull", "-q"], check=True)
sys.path.insert(0, "congress-speech")
!git -C congress-speech log --oneline -1

## 3. Install what Colab lacks
MLflow is pinned to the server's version.

In [ ]:
%pip install -q "mlflow==3.14.0" optuna

## 4. Connect to MLflow and Hugging Face
Read the compute-unit rate for this runtime from Colab's resources panel and type it below; it is stored on every run as `cu_estimate`.

In [ ]:
CU_PER_HOUR = None   # e.g. 4.8 for an L4: read it from the resources panel
assert CU_PER_HOUR, "set CU_PER_HOUR from Colab's resources panel"

os.environ["MLFLOW_TRACKING_URI"] = "https://mlflow.withinnoise.dev"
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("MLFLOW_USER")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("MLFLOW_PASSWORD")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from huggingface_hub import whoami
try:
    print("Hugging Face user:", whoami(token=os.environ["HF_TOKEN"])["name"])
except Exception as e:
    raise SystemExit(f"HF_TOKEN is not a valid token ({type(e).__name__}); make a Write token and update the secret") from None

import torch, transformers
from congress_speech import tracking, finetune, stats
mlflow = tracking.setup()
COMMIT = subprocess.run(["git", "-C", "congress-speech", "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
RUNTIME = "colab-" + torch.cuda.get_device_name(0).split()[-1]
print(RUNTIME, "torch", torch.__version__, "transformers", transformers.__version__, "commit", COMMIT)

## 5. Data and a dry run

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

def sample(congress):
    path = hf_hub_download("kvenanzi/congressional-record-speeches-1995-2026", f"samples/govinfo_{congress}.parquet", repo_type="dataset")
    return pd.read_parquet(path)

In [ ]:
# A tiny run first, so that a mistake costs a minute rather than compute units. Not logged.
s = sample(108)
tiny = pd.concat([g.head(n) for (sp, g), n in zip(s.groupby("split"), [32, 64, 32])])  # test, train, val
_, _, r, pred = finetune.train(tiny, finetune.Params(epochs=1, max_len=128, batch_size=16))
print(r["val_acc"], len(pred), pred.columns.tolist())

## 6. The study

In [ ]:
import optuna, time
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend, JournalFileOpenLock
from google.colab import drive

drive.mount("/content/drive")
JOURNAL = "/content/drive/MyDrive/optuna/studies.log"
os.makedirs(os.path.dirname(JOURNAL), exist_ok=True)
storage = JournalStorage(JournalFileBackend(JOURNAL, lock_obj=JournalFileOpenLock(JOURNAL)))

STUDY, N_TRIALS, DEV_CONGRESS = "congress-speech-sweep-01", 15, 108
study = optuna.create_study(study_name=STUDY, storage=storage, direction="maximize", load_if_exists=True,
                            sampler=optuna.samplers.TPESampler(seed=20261105))
data = sample(DEV_CONGRESS)

def objective(trial):
    p = finetune.Params(
        lr=trial.suggest_float("lr", 1e-5, 1e-4, log=True),
        epochs=trial.suggest_categorical("epochs", [1, 2, 3]),
        warmup=trial.suggest_float("warmup", 0.0, 0.1),
        max_len=trial.suggest_categorical("max_len", [256, 512]),
        batch_size=trial.suggest_categorical("batch_size", [16, 32]),
        weight_decay=trial.suggest_float("weight_decay", 0.0, 0.1),
    )
    t0 = time.time()
    with mlflow.start_run(run_name=f"trial-{trial.number:03d}", nested=True):
        mlflow.set_tags({"optuna.study": STUDY, "optuna.trial": trial.number, "congress": DEV_CONGRESS,
                         "model": "modernbert", "stage": "sweep", "runtime": RUNTIME, "git_commit": COMMIT})
        mlflow.log_params(vars(p))
        _, _, r, _ = finetune.train(data, p, log=lambda m, step: mlflow.log_metrics(m, step=step), eval_test=False)
        hours = (time.time() - t0) / 3600
        mlflow.log_metrics({"val_acc": r["val_acc"], "hours": hours})
        mlflow.set_tag("cu_estimate", round(hours * CU_PER_HOUR, 2))
    torch.cuda.empty_cache()
    return r["val_acc"]

parent_id = study.user_attrs.get("mlflow_parent_run_id")
with mlflow.start_run(run_id=parent_id, run_name=STUDY) as parent:
    if parent_id is None:
        study.set_user_attr("mlflow_parent_run_id", parent.info.run_id)
        mlflow.set_tags({"optuna.study": STUDY, "stage": "sweep-parent", "congress": DEV_CONGRESS})
    remaining = N_TRIALS - len([t for t in study.trials if t.state.is_finished()])
    if remaining > 0:
        study.optimize(objective, n_trials=remaining)
    best = {"best_trial": study.best_trial.number, "best_params": study.best_params, "best_val_acc": study.best_value}
    mlflow.log_metric("best_val_acc", study.best_value)
    mlflow.log_dict(best, "optuna/best.json")
    from optuna import visualization as vis
    for name, plot in {"optimization_history": vis.plot_optimization_history, "param_importances": vis.plot_param_importances,
                       "parallel_coordinate": vis.plot_parallel_coordinate, "slice": vis.plot_slice}.items():
        try:
            mlflow.log_figure(plot(study), f"optuna/{name}.html")
        except Exception as e:
            print("skipped", name, e)
    print(best)
    print(f"{os.environ['MLFLOW_TRACKING_URI']}/#/experiments/{parent.info.experiment_id}/runs/{parent.info.run_id}")